# Tackling Mode Collapse in GANs: DCGAN vs WGAN-GP
**Platform:** Kaggle | **Accelerator:** GPU T4 x2  
**Dataset:** Anime Faces 64×64 or Pokemon Sprites

## 1. Environment Setup & Imports

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import csv

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, utils as vutils

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Device setup — use both GPUs if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Using device: {device}')
print(f'Number of GPUs: {n_gpus}')
if torch.cuda.is_available():
    for i in range(n_gpus):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

def log_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 2)
        reserved = torch.cuda.memory_reserved() / (1024 ** 2)
        print(f'  [GPU RAM] Allocated: {allocated:.1f} MB | Reserved: {reserved:.1f} MB')


## 2. Hyperparameters

In [ ]:
# ── Shared ──────────────────────────────────────────────
IMAGE_SIZE   = 64
NC           = 3          # number of channels (RGB)
NZ           = 100        # latent vector size
NGF          = 64         # generator feature map base
NDF          = 64         # discriminator feature map base
BATCH_SIZE   = 64
import platform
NUM_WORKERS  = 0 if platform.system() == 'Windows' else 2
LR           = 0.0002
BETAS        = (0.5, 0.999)

# ── Training epochs ─────────────────────────────────────
DCGAN_EPOCHS  = 50
WGANGP_EPOCHS = 50

# ── WGAN-GP specific ────────────────────────────────────
LAMBDA_GP     = 10        # gradient penalty weight
CRITIC_ITERS  = 5         # critic updates per generator update

# ── Checkpoint & output dirs ────────────────────────────
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/outputs',     exist_ok=True)
CKPT_INTERVAL = 10        # save every N epochs

print('Hyperparameters set.')

## 3. Data Preparation

In [ ]:
# ── Locate dataset ──────────────────────────────────────
# Kaggle mounts datasets under /kaggle/input/
# Try anime-faces first, fall back to pokemon-sprites
ANIME_PATH   = '/kaggle/input/datasets/soumikrakshit/anime-faces/data'
POKEMON_PATH = '/kaggle/input/pokemon-sprites'

if os.path.exists(ANIME_PATH):
    DATA_ROOT = ANIME_PATH
    print(f'Using Anime Faces dataset: {DATA_ROOT}')
elif os.path.exists(POKEMON_PATH):
    DATA_ROOT = POKEMON_PATH
    print(f'Using Pokemon Sprites dataset: {DATA_ROOT}')
else:
    # Fallback: scan /kaggle/input/ for any image folder
    DATA_ROOT = None
    for root, dirs, files in os.walk('/kaggle/input'):
        imgs = [f for f in files if f.lower().endswith(('.png','.jpg','.jpeg'))]
        if len(imgs) > 100:
            DATA_ROOT = root
            print(f'Auto-detected dataset at: {DATA_ROOT} ({len(imgs)} images)')
            break
    if DATA_ROOT is None:
        raise FileNotFoundError('No dataset found under /kaggle/input/. Please add the dataset to this notebook.')

# Count images
all_images = []
for root, _, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            all_images.append(os.path.join(root, f))
print(f'Total images found: {len(all_images)}')

In [ ]:
class ImageDataset(Dataset):
    """Generic image dataset that loads from a flat or nested folder."""
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            with Image.open(self.image_paths[idx]) as f_img:
                img = f_img.convert('RGB')
        except Exception:
            # Return a blank image if file is corrupt
            img = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), 0)
        if self.transform:
            img = self.transform(img)
        return img


transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],   # → [-1, 1]
                         [0.5, 0.5, 0.5]),
])

dataset    = ImageDataset(all_images, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=False, drop_last=True)

print(f'Dataset size : {len(dataset)}')
print(f'Batches/epoch: {len(dataloader)}')

# Visualise a sample batch
sample_batch = next(iter(dataloader))
grid = vutils.make_grid(sample_batch[:16], nrow=4, normalize=True, value_range=(-1,1))
plt.figure(figsize=(8,8))
plt.title('Sample Training Images')
plt.imshow(grid.permute(1,2,0).cpu().numpy())
plt.axis('off')
plt.savefig('/kaggle/working/outputs/sample_training_images.png', bbox_inches='tight')
plt.show()
plt.close('all')

## 4. Model Architectures

In [ ]:
def weights_init(m):
    """DCGAN paper weight initialisation: N(0, 0.02)."""
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


# ── Generator (shared by DCGAN & WGAN-GP) ───────────────
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(NZ,  NGF*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*8),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*4),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*2),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF*2, NGF,   4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF),
            nn.ReLU(True),
            nn.ConvTranspose2d(NGF,   NC,    4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.main(z)


# ── DCGAN Discriminator (MODIFIED: Sigmoid output removed for AMP) ─
class DCGANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(NC,    NDF,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF,   NDF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*2),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*4),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*8),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*8, 1,     4, 1, 0, bias=False),
            # nn.Sigmoid() <- REMOVED! BCEWithLogitsLoss handles this internally now
        )

    def forward(self, x):
        return self.main(x).view(-1)


# ── WGAN-GP Critic (no Sigmoid) ──────────────────────────
class WGANCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(NC,    NDF,   4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF,   NDF*2, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(NDF*2, affine=True),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(NDF*4, affine=True),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(NDF*8, affine=True),
            nn.LeakyReLU(0.2, True),
            nn.Conv2d(NDF*8, 1,     4, 1, 0, bias=False)
        )

    def forward(self, x):
        return self.main(x).view(-1)

print('Model classes defined.')


## 5. DCGAN Training

In [ ]:
# ── Initialise models ───────────────────────────────────
netG_dc = Generator().to(device)
netD_dc = DCGANDiscriminator().to(device)

if n_gpus > 1:
    netG_dc = nn.DataParallel(netG_dc)
    netD_dc = nn.DataParallel(netD_dc)

netG_dc.apply(weights_init)
netD_dc.apply(weights_init)

# Loss & optimisers (MODIFIED: Use BCEWithLogitsLoss)
criterion_dc  = nn.BCEWithLogitsLoss()
optG_dc = optim.Adam(netG_dc.parameters(), lr=LR, betas=BETAS)
optD_dc = optim.Adam(netD_dc.parameters(), lr=LR, betas=BETAS)

# Mixed precision scaler
scaler_dc = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# Fixed noise for consistent visualisation
fixed_noise = torch.randn(64, NZ, 1, 1, device=device)

# Loss log
dcgan_G_losses, dcgan_D_losses = [], []

print('DCGAN initialised.')
print(f'  Generator params     : {sum(p.numel() for p in netG_dc.parameters()):,}')
print(f'  Discriminator params : {sum(p.numel() for p in netD_dc.parameters()):,}')


In [ ]:
print('=' * 60)
print('Starting DCGAN Training')
print('=' * 60)

for epoch in range(1, DCGAN_EPOCHS + 1):
    epoch_G_loss, epoch_D_loss = 0.0, 0.0

    for i, real_imgs in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        b_size    = real_imgs.size(0)

        real_label = torch.ones(b_size,  device=device)
        fake_label = torch.zeros(b_size, device=device)

        # ── Train Discriminator ──────────────────────────
        netD_dc.zero_grad()
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            out_real = netD_dc(real_imgs)
            loss_D_real = criterion_dc(out_real, real_label)

            noise     = torch.randn(b_size, NZ, 1, 1, device=device)
            fake_imgs = netG_dc(noise).detach()
            out_fake  = netD_dc(fake_imgs)
            loss_D_fake = criterion_dc(out_fake, fake_label)

            loss_D = loss_D_real + loss_D_fake

        scaler_dc.scale(loss_D).backward()
        scaler_dc.step(optD_dc)
        scaler_dc.update()

        # ── Train Generator ──────────────────────────────
        netG_dc.zero_grad()
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            noise     = torch.randn(b_size, NZ, 1, 1, device=device)
            fake_imgs = netG_dc(noise)
            out_fake  = netD_dc(fake_imgs)
            loss_G    = criterion_dc(out_fake, real_label)   # generator wants D to output 1

        scaler_dc.scale(loss_G).backward()
        scaler_dc.step(optG_dc)
        scaler_dc.update()

        epoch_G_loss += loss_G.item()
        epoch_D_loss += loss_D.item()

    avg_G = epoch_G_loss / len(dataloader)
    avg_D = epoch_D_loss / len(dataloader)
    dcgan_G_losses.append(avg_G)
    dcgan_D_losses.append(avg_D)

    print(f'[DCGAN] Epoch {epoch:03d}/{DCGAN_EPOCHS} | Loss_D: {avg_D:.4f} | Loss_G: {avg_G:.4f}')
    log_gpu_memory()

    # Save checkpoint every N epochs
    if epoch % CKPT_INTERVAL == 0:
        torch.save(netG_dc.state_dict(), f'/kaggle/working/checkpoints/dcgan_G_epoch{epoch}.pth')
        torch.save(netD_dc.state_dict(), f'/kaggle/working/checkpoints/dcgan_D_epoch{epoch}.pth')

        # Save generated grid
        with torch.no_grad():
            fake = netG_dc(fixed_noise)
        grid = vutils.make_grid(fake[:16], nrow=4, normalize=True, value_range=(-1,1))
        plt.figure(figsize=(6,6))
        plt.title(f'DCGAN — Epoch {epoch}')
        plt.imshow(grid.permute(1,2,0).cpu().numpy())
        plt.axis('off')
        plt.savefig(f'/kaggle/working/outputs/dcgan_epoch{epoch:03d}.png', bbox_inches='tight')
        plt.show()
        plt.close('all')

print('DCGAN Training Complete!')


## 6. WGAN-GP Training

In [ ]:
def compute_gradient_penalty(critic, real_imgs, fake_imgs, device):
    """Calculates the gradient penalty for WGAN-GP."""
    batch_size = real_imgs.size(0)
    # Random interpolation coefficient
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = (alpha * real_imgs + (1 - alpha) * fake_imgs).requires_grad_(True)

    d_interpolated = critic(interpolated)

    # Compute gradients
    gradients = torch.autograd.grad(
        outputs=d_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(d_interpolated, device=device),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients  = gradients.view(batch_size, -1)
    grad_norm  = gradients.norm(2, dim=1)
    gp         = ((grad_norm - 1) ** 2).mean()
    return gp


# ── Initialise WGAN-GP models ────────────────────────────
netG_wgan = Generator().to(device)
netC_wgan = WGANCritic().to(device)

# DataParallel disabled for WGAN-GP to prevent torch.autograd.grad issues
# if n_gpus > 1:
#     netG_wgan = nn.DataParallel(netG_wgan)
#     netC_wgan = nn.DataParallel(netC_wgan)

netG_wgan.apply(weights_init)
netC_wgan.apply(weights_init)

# Note: betas=(0.5, 0.999) used as required by assignment, but betas=(0.0, 0.9) is theoretically recommended.
optG_wgan = optim.Adam(netG_wgan.parameters(), lr=LR, betas=BETAS)
optC_wgan = optim.Adam(netC_wgan.parameters(), lr=LR, betas=BETAS)

# Mixed precision scaler for WGAN-GP
scaler_wgan = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

# Loss log
wgan_G_losses, wgan_C_losses = [], []

print('WGAN-GP initialised.')
print(f'  Generator params : {sum(p.numel() for p in netG_wgan.parameters()):,}')
print(f'  Critic params    : {sum(p.numel() for p in netC_wgan.parameters()):,}')

In [ ]:
print('=' * 60)
print('Starting WGAN-GP Training')
print('=' * 60)

for epoch in range(1, WGANGP_EPOCHS + 1):
    epoch_G_loss, epoch_C_loss = 0.0, 0.0
    gen_updates = 0

    data_iter = iter(dataloader)

    while True:
        # ── Train Critic CRITIC_ITERS times ──────────────
        exhausted = False
        exhausted = False
        exhausted = False
        for _ in range(CRITIC_ITERS):
            try:
                real_imgs = next(data_iter).to(device)
            except StopIteration:
                exhausted = True
                break
                real_imgs = next(data_iter).to(device)

            b_size = real_imgs.size(0)
            netC_wgan.zero_grad()

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                noise     = torch.randn(b_size, NZ, 1, 1, device=device)
                fake_imgs = netG_wgan(noise).detach()
                score_real = netC_wgan(real_imgs)
                score_fake = netC_wgan(fake_imgs)

            gp     = compute_gradient_penalty(netC_wgan, real_imgs.float(), fake_imgs.float(), device)
            
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                loss_C = score_fake.mean() - score_real.mean() + LAMBDA_GP * gp

            scaler_wgan.scale(loss_C).backward()
            scaler_wgan.step(optC_wgan)
            scaler_wgan.update()
            epoch_C_loss += loss_C.item()

        # ── Train Generator once ─────────────────────────
        if exhausted:
            break
        if exhausted:
            break
        netG_wgan.zero_grad()
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            noise     = torch.randn(BATCH_SIZE, NZ, 1, 1, device=device)
            fake_imgs = netG_wgan(noise)
            loss_G    = -netC_wgan(fake_imgs).mean()

        scaler_wgan.scale(loss_G).backward()
        scaler_wgan.step(optG_wgan)
        scaler_wgan.update()
        epoch_G_loss += loss_G.item()
        gen_updates  += 1

        if gen_updates >= len(dataloader) // CRITIC_ITERS:
            break

    avg_G = epoch_G_loss / max(gen_updates, 1)
    avg_C = epoch_C_loss / max(gen_updates * CRITIC_ITERS, 1)
    wgan_G_losses.append(avg_G)
    wgan_C_losses.append(avg_C)

    print(f'[WGAN-GP] Epoch {epoch:03d}/{WGANGP_EPOCHS} | Loss_C: {avg_C:.4f} | Loss_G: {avg_G:.4f}')
    log_gpu_memory()

    # Save checkpoint every N epochs
    if epoch % CKPT_INTERVAL == 0:
        torch.save(netG_wgan.state_dict(),
                   f'/kaggle/working/checkpoints/wgangp_G_epoch{epoch}.pth')
        torch.save(netC_wgan.state_dict(),
                   f'/kaggle/working/checkpoints/wgangp_C_epoch{epoch}.pth')

        with torch.no_grad():
            fake = netG_wgan(fixed_noise)
        grid = vutils.make_grid(fake[:16], nrow=4, normalize=True, value_range=(-1,1))
        plt.figure(figsize=(6,6))
        plt.title(f'WGAN-GP — Epoch {epoch}')
        plt.imshow(grid.permute(1,2,0).cpu().numpy())
        plt.axis('off')
        plt.savefig(f'/kaggle/working/outputs/wgangp_epoch{epoch:03d}.png', bbox_inches='tight')
        plt.show()
        plt.close('all')

print('WGAN-GP Training Complete!')

## 7. Visualisation & Comparison

In [ ]:
# ── Loss Curves ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(dcgan_G_losses, label='Generator',     color='royalblue')
axes[0].plot(dcgan_D_losses, label='Discriminator', color='tomato')
axes[0].set_title('DCGAN — Training Losses')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(wgan_G_losses, label='Generator', color='royalblue')
axes[1].plot(wgan_C_losses, label='Critic',    color='tomato')
axes[1].set_title('WGAN-GP — Training Losses')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/loss_curves.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close('all')
print('Loss curves saved.')

In [ ]:
# ── Final Generated Samples — DCGAN (10 images) ─────────
netG_dc.eval()
with torch.no_grad():
    test_noise = torch.randn(10, NZ, 1, 1, device=device)
    dcgan_samples = netG_dc(test_noise).cpu()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('DCGAN — Final Generated Samples', fontsize=16, fontweight='bold')
for i, ax in enumerate(axes.flat):
    img = dcgan_samples[i].permute(1,2,0).numpy()
    img = (img * 0.5 + 0.5).clip(0, 1)   # denormalise
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Sample {i+1}', fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/dcgan_final_samples.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close('all')
print('DCGAN final samples saved.')

In [ ]:
# ── Final Generated Samples — WGAN-GP (10 images) ───────
netG_wgan.eval()
with torch.no_grad():
    wgan_samples = netG_wgan(test_noise).cpu()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('WGAN-GP — Final Generated Samples', fontsize=16, fontweight='bold')
for i, ax in enumerate(axes.flat):
    img = wgan_samples[i].permute(1,2,0).numpy()
    img = (img * 0.5 + 0.5).clip(0, 1)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Sample {i+1}', fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/wgangp_final_samples.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close('all')
print('WGAN-GP final samples saved.')

In [ ]:
# ── Side-by-Side Comparison ──────────────────────────────
# Same 8 noise vectors fed to both generators
compare_noise = torch.randn(8, NZ, 1, 1, device=device)

with torch.no_grad():
    dc_out   = netG_dc(compare_noise).cpu()
    wgan_out = netG_wgan(compare_noise).cpu()

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('Side-by-Side Comparison: DCGAN (top 2 rows) vs WGAN-GP (bottom 2 rows)',
             fontsize=14, fontweight='bold')

for col in range(8):
    # DCGAN row 1 & 2 (same image, just filling both rows for visual weight)
    for row in range(2):
        img = dc_out[col].permute(1,2,0).numpy()
        img = (img * 0.5 + 0.5).clip(0,1)
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if row == 0:
            axes[row][col].set_title(f'z_{col+1}', fontsize=8)
    # WGAN-GP row 3 & 4
    for row in range(2, 4):
        img = wgan_out[col].permute(1,2,0).numpy()
        img = (img * 0.5 + 0.5).clip(0,1)
        axes[row][col].imshow(img)
        axes[row][col].axis('off')

# Row labels
axes[0][0].set_ylabel('DCGAN', fontsize=12, rotation=90, labelpad=40)
axes[2][0].set_ylabel('WGAN-GP', fontsize=12, rotation=90, labelpad=40)

plt.tight_layout()
plt.savefig('/kaggle/working/outputs/comparison.png', bbox_inches='tight', dpi=150)
plt.show()
plt.close('all')
print('Comparison image saved.')

## 8. Save Final Checkpoints & Loss Logs

In [ ]:
# Save final model weights
torch.save(netG_dc.state_dict(),   '/kaggle/working/checkpoints/dcgan_G_final.pth')
torch.save(netD_dc.state_dict(),   '/kaggle/working/checkpoints/dcgan_D_final.pth')
torch.save(netG_wgan.state_dict(), '/kaggle/working/checkpoints/wgangp_G_final.pth')
torch.save(netC_wgan.state_dict(), '/kaggle/working/checkpoints/wgangp_C_final.pth')

# Save loss log as CSV
with open('/kaggle/working/outputs/loss_log.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'dcgan_G', 'dcgan_D', 'wgangp_G', 'wgangp_C'])
    for i in range(max(len(dcgan_G_losses), len(wgan_G_losses))):
        writer.writerow([
            i + 1,
            dcgan_G_losses[i]  if i < len(dcgan_G_losses) else '',
            dcgan_D_losses[i]  if i < len(dcgan_D_losses) else '',
            wgan_G_losses[i]   if i < len(wgan_G_losses)  else '',
            wgan_C_losses[i]   if i < len(wgan_C_losses)  else '',
        ])

print('All checkpoints and logs saved!')
print('\nOutput files:')
for f in sorted(os.listdir('/kaggle/working/outputs')):
    size = os.path.getsize(f'/kaggle/working/outputs/{f}')
    print(f'  {f:45s} {size/1024:8.1f} KB')

## 9. Summary

| | DCGAN | WGAN-GP |
|---|---|---|
| **Loss Function** | Binary Cross Entropy | Wasserstein + Gradient Penalty |
| **Output Layer** | Sigmoid | No activation (raw scores) |
| **Normalisation** | Batch Norm | Instance Norm |
| **Critic Updates** | 1 per G update | 5 per G update |
| **Gradient Penalty** | ❌ | ✅ λ=10 |
| **Mixed Precision** | ✅ | ✅ |
| **Mode Collapse Risk** | High | Low |
| **Training Stability** | Moderate | High |

## 10. Inference Prototype (App Backend Testing)
This cell simulates what your Gradio/Streamlit app will do. It tests loading the saved `.pth` weights from disk into a fresh Generator, outside of the training loop. You can copy `generate_inference_images` directly into your future web app!

In [ ]:
def generate_inference_images(model_choice, num_images=4, device='cpu'):
    """
    Simulates the backend of your Gradio/Streamlit app.
    Loads the weights from disk, generates images, and returns them as a numpy array.
    """
    # 1. Instantiate a fresh Generator (which we defined earlier)
    generator = Generator().to(device)
    
    # 2. Select the correct checkpoint path
    if model_choice.lower() == 'dcgan':
        ckpt_path = '/kaggle/working/checkpoints/dcgan_G_final.pth'
    else:
        ckpt_path = '/kaggle/working/checkpoints/wgangp_G_final.pth'
        
    # 3. Load weights (handling newer/older PyTorch versions seamlessly)
    try:
        generator.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    except TypeError:
        generator.load_state_dict(torch.load(ckpt_path, map_location=device))
        
    generator.eval()
    
    # 4. Generate images from pure noise
    with torch.no_grad():
        noise = torch.randn(num_images, NZ, 1, 1, device=device)
        generated_tensors = generator(noise).cpu()
    
    # 5. Format for display (Gradio usually expects a PIL Image or Numpy Array)
    # We use make_grid to tile the images beautifully
    nrow = int(np.sqrt(num_images))
    grid = vutils.make_grid(generated_tensors, nrow=nrow, normalize=True, value_range=(-1,1))
    image_np = (grid.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    
    return image_np

# --- Test the Inference Function ---
plt.figure(figsize=(10, 5))

# Test DCGAN Checkpoint
plt.subplot(1, 2, 1)
dcgan_output = generate_inference_images('dcgan', num_images=16, device=device)
plt.imshow(dcgan_output)
plt.title("Loaded DCGAN Checkpoint")
plt.axis('off')

# Test WGAN-GP Checkpoint
plt.subplot(1, 2, 2)
wgan_output = generate_inference_images('wgan-gp', num_images=16, device=device)
plt.imshow(wgan_output)
plt.title("Loaded WGAN-GP Checkpoint")
plt.axis('off')

plt.tight_layout()
plt.show()
plt.close('all')

print("Inference test successful! These images were generated purely by loading the .pth files.")

In [ ]:
# ── Gradio Interface for Model Comparison ──────────────────
import gradio as gr

def generate_images_ui(model_choice, num_samples):
    """
    Gradio UI for generating and comparing images from both models.
    Returns a grid of generated images.
    """
    num_samples = min(int(num_samples), 16)
    
    # Generate images based on selected model
    if model_choice == "DCGAN":
        netG_dc.eval()
        with torch.no_grad():
            noise = torch.randn(num_samples, NZ, 1, 1, device=device)
            images = netG_dc(noise).cpu()
    else:  # WGAN-GP
        netG_wgan.eval()
        with torch.no_grad():
            noise = torch.randn(num_samples, NZ, 1, 1, device=device)
            images = netG_wgan(noise).cpu()
    
    # Create grid visualization
    nrow = int(np.sqrt(num_samples))
    grid = vutils.make_grid(images, nrow=nrow, normalize=True, value_range=(-1,1))
    grid_np = grid.permute(1,2,0).numpy()
    
    # Denormalize to [0, 1] for display
    grid_np = (grid_np * 0.5 + 0.5).clip(0, 1)
    
    return grid_np

# Create Gradio interface
demo = gr.Interface(
    fn=generate_images_ui,
    inputs=[
        gr.Dropdown(
            ["DCGAN", "WGAN-GP"], 
            value="DCGAN",
            label="Select GAN Model"
        ),
        gr.Slider(
            minimum=4, 
            maximum=16, 
            step=4, 
            value=8,
            label="Number of Samples to Generate"
        )
    ],
    outputs=gr.Image(label="Generated Images"),
    title="GAN Comparison: DCGAN vs WGAN-GP",
    description="Compare baseline DCGAN with improved WGAN-GP trained on Anime Faces dataset.

**DCGAN**: Baseline approach with mode collapse risk.
**WGAN-GP**: Improved stability with gradient penalty for diverse outputs.",
    examples=[
        ["DCGAN", 4],
        ["DCGAN", 8],
        ["WGAN-GP", 4],
        ["WGAN-GP", 8],
    ],
    allow_flagging="never"
)

# Launch the app (uncomment to run)
# demo.launch(share=True)
print("Gradio interface ready! Uncomment demo.launch(share=True) to deploy.")
